***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [第 2 章：干涉测量的数学工具箱](2_0_introduction.ipynb)
    * 上一节：[2.9 采样理论：离散测量的分辨率与混叠](2_9_sampling_theory.ipynb)
    * 下一节：[2.11 最小二乘与参数估计](2_11_least_squares.ipynb)

***


导入标准模块。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    from IPython.display import HTML
except ImportError:
    def HTML(*args, **kwargs):
        return None
%matplotlib inline
HTML('../style/course.css')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12


## 2.10 线性代数：从测量方程到矩阵表示<a id='math:sec:lin_alg'></a>

前面几节把信号表示、傅里叶变换、采样和相关逐步建立起来；从这里开始，需要把这些操作组织成有限维方程。可见度数据可以堆叠成向量，图像像素可以堆叠成向量，傅里叶采样、加权、卷积和校准都可以写成矩阵或矩阵的组合。这样一来，成像和定标就不再只是连续公式，而成为可以求解、诊断和优化的线性代数问题。

本节不追求完整的线性代数教程，而是围绕射电干涉测量中最常见的对象展开：向量、内积、外积、矩阵乘法、特殊矩阵乘积、加权线性系统、正规方程，以及卷积矩阵和测量矩阵的结构。


### 2.10.1 向量、内积与外积<a id='math:sec:lin_alg_vectors'></a>

本书默认向量为列向量。一个 $n$ 维复列向量写作

<a id='math:eq:column_vector'></a><!--\label{math:eq:column_vector}-->
$$
\mathbf{v}=\begin{pmatrix}v_1\\v_2\\\vdots\\v_n\end{pmatrix}\in\mathbb{C}^n.
$$

行向量通常写成转置形式 $\mathbf{v}^T$，而在复向量空间中更常用的是 Hermitian 转置 $\mathbf{v}^H=(\mathbf{v}^*)^T$。默认列向量的好处，是所有线性算子都可以统一写成从左侧作用：

$$
\mathbf{b}=\mathbf{A}\mathbf{x}.
$$

这里 $\mathbf{x}$ 是未知量向量，$\mathbf{A}$ 是前向算子，$\mathbf{b}$ 是观测数据向量。


复向量的内积定义为

<a id='math:eq:hermitian_inner_product'></a><!--\label{math:eq:hermitian_inner_product}-->
$$
\langle \mathbf{u},\mathbf{v}\rangle
=\mathbf{u}^H\mathbf{v}
=\sum_{i=1}^n u_i^*v_i.
$$

它把两个向量映射成一个复数。内积的模表示相似程度，辐角表示相对相位；这与相关器中复共轭乘积的结构完全一致。向量范数由内积给出：

$$
\|\mathbf{v}\|=\sqrt{\mathbf{v}^H\mathbf{v}}.
$$

对实向量，内积退化为熟悉的点积，并给出投影长度：

$$
\mathbf{b}\cdot\hat{\mathbf{a}}=\|\mathbf{b}\|\cos\theta.
$$

![点积投影](figures/projection.png)

**图 2.10.1** 点积的投影解释。内积不仅是代数运算，也是在某个方向上提取分量的操作。


外积把两个向量变成一个矩阵：

<a id='math:eq:outer_product'></a><!--\label{math:eq:outer_product}-->
$$
\mathbf{u}\mathbf{v}^H
=\begin{pmatrix}u_1v_1^*&u_1v_2^*&\cdots\\
u_2v_1^*&u_2v_2^*&\cdots\\
\vdots&\vdots&\ddots
\end{pmatrix}.
$$

若 $\hat{\mathbf{u}}$ 与 $\hat{\mathbf{v}}$ 是单位向量，则矩阵 $\hat{\mathbf{u}}\hat{\mathbf{v}}^H$ 作用在任意向量 $\mathbf{x}$ 上时，先取出 $\mathbf{x}$ 在 $\hat{\mathbf{v}}$ 方向上的分量，再把该分量放到 $\hat{\mathbf{u}}$ 方向：

$$
(\hat{\mathbf{u}}\hat{\mathbf{v}}^H)\mathbf{x}
=\hat{\mathbf{u}}(\hat{\mathbf{v}}^H\mathbf{x}).
$$

因此，外积是构造秩 1 线性算子的基本方式。协方差矩阵、相干矩阵、极化 coherency matrix 和某些方向响应，都可以用外积来理解。


### 2.10.2 矩阵作为线性映射<a id='math:sec:lin_alg_matrices'></a>

矩阵

<a id='math:eq:matrix_definition'></a><!--\label{math:eq:matrix_definition}-->
$$
\mathbf{A}=\begin{pmatrix}
a_{11}&a_{12}&\cdots&a_{1n}\\
a_{21}&a_{22}&\cdots&a_{2n}\\
\vdots&\vdots&\ddots&\vdots\\
a_{m1}&a_{m2}&\cdots&a_{mn}
\end{pmatrix}\in\mathbb{C}^{m\times n}
$$

表示一个从 $\mathbb{C}^n$ 到 $\mathbb{C}^m$ 的线性映射。矩阵乘以向量时，第 $i$ 个输出为

$$
(\mathbf{A}\mathbf{x})_i=\sum_{j=1}^n a_{ij}x_j.
$$

这说明每一行都是一个对输入向量的线性测量。干涉测量中的测量矩阵也可这样理解：每一行对应一个基线、时间和频率样本对天空像素的响应。


常用矩阵操作包括转置、复共轭和 Hermitian 转置：

<a id='math:eq:matrix_transposes'></a><!--\label{math:eq:matrix_transposes}-->
$$
(\mathbf{A}^T)_{ij}=a_{ji},
\qquad
(\mathbf{A}^*)_{ij}=a_{ij}^*,
\qquad
\mathbf{A}^H=(\mathbf{A}^T)^*.
$$

在加权内积和正规方程中，Hermitian 转置最常出现。若 $\mathbf{A}$ 是方阵，并且存在 $\mathbf{A}^{-1}$ 使得 $\mathbf{A}^{-1}\mathbf{A}=\mathbf{A}\mathbf{A}^{-1}=\mathbf{I}$，则称其可逆。满足 $\mathbf{A}=\mathbf{A}^H$ 的矩阵称为 Hermitian 矩阵；满足 $\mathbf{U}^H\mathbf{U}=\mathbf{I}$ 的矩阵称为酉矩阵。DFT 矩阵在适当归一化后就是酉矩阵，这也是它保持能量和内积结构的原因。

对角矩阵 $\mathrm{diag}(\mathbf{w})$ 常用来表示逐点权重；向量化 $\mathrm{vec}(\mathbf{A})$ 则把矩阵按列堆叠成一个长向量。极化和校准推导中经常需要在矩阵形式与向量形式之间来回转换。


### 2.10.3 几种常见矩阵乘积<a id='math:sec:matrix_products'></a>

普通矩阵乘法把列与行做内积：

<a id='math:eq:matrix_product'></a><!--\label{math:eq:matrix_product}-->
$$
(\mathbf{A}\mathbf{D})_{ij}=\sum_{k=1}^n a_{ik}d_{kj}.
$$

Hadamard 积是逐元素乘法：

<a id='math:eq:hadamard_product'></a><!--\label{math:eq:hadamard_product}-->
$$
(\mathbf{A}\odot\mathbf{C})_{ij}=a_{ij}c_{ij}.
$$

它适合表达权重、掩膜、逐基线增益或逐像素修正。Kronecker 积把两个矩阵扩展成一个更大的块矩阵：

<a id='math:eq:kronecker_product'></a><!--\label{math:eq:kronecker_product}-->
$$
\mathbf{A}\otimes\mathbf{B}
=\begin{pmatrix}
a_{11}\mathbf{B}&a_{12}\mathbf{B}&\cdots\\
a_{21}\mathbf{B}&a_{22}\mathbf{B}&\cdots\\
\vdots&\vdots&\ddots
\end{pmatrix}.
$$

它常用于把两个线性空间组合起来，例如天线响应与极化响应。Khatri-Rao 积可理解为按列或按行组织的一组 Kronecker 积；不同文献约定略有差异。本书在需要时会明确说明采用的排列方式。


向量化恒等式在极化和校准文献中很常见，其中最重要的一条是

<a id='math:eq:vec_identity'></a><!--\label{math:eq:vec_identity}-->
$$
\mathrm{vec}(\mathbf{A}\mathbf{X}\mathbf{B})
=(\mathbf{B}^T\otimes\mathbf{A})\mathrm{vec}(\mathbf{X}).
$$

它的意义是把一个矩阵方程改写成标准的矩阵-向量方程。对 Jones 矩阵作用在 coherency matrix 两侧的形式，常可用这类恒等式转成 Mueller 矩阵作用在向量化 Stokes/coherency 表示上的形式。这里不要求手工展开所有分量，但需要理解它把“左右夹乘”变成了“一个大矩阵左乘”。


### 2.10.4 加权线性系统与正规方程<a id='math:sec:lin_alg_lin_sys'></a>

许多离散化后的测量问题都可以写成

<a id='math:eq:linear_measurement_model'></a><!--\label{math:eq:linear_measurement_model}-->
$$
\mathbf{b}=\mathbf{A}\mathbf{x}+\boldsymbol{\epsilon}.
$$

这里 $\mathbf{x}$ 是未知参数，$\mathbf{A}$ 是前向算子，$\mathbf{b}$ 是观测数据，$\boldsymbol{\epsilon}$ 是噪声。若噪声为零均值高斯噪声，协方差为 $\boldsymbol{\Sigma}$，则最大似然估计等价于最小化加权平方残差

<a id='math:eq:weighted_chi_square'></a><!--\label{math:eq:weighted_chi_square}-->
$$
\chi^2(\mathbf{x})=(\mathbf{A}\mathbf{x}-\mathbf{b})^H
\mathbf{W}(\mathbf{A}\mathbf{x}-\mathbf{b}),
\qquad
\mathbf{W}=\boldsymbol{\Sigma}^{-1}.
$$

令梯度为零，得到正规方程

<a id='math:eq:normal_equations'></a><!--\label{math:eq:normal_equations}-->
$$
\mathbf{A}^H\mathbf{W}\mathbf{A}\,\hat{\mathbf{x}}
=\mathbf{A}^H\mathbf{W}\mathbf{b}.
$$

若 $\mathbf{A}^H\mathbf{W}\mathbf{A}$ 非奇异，形式解为

$$
\hat{\mathbf{x}}=(\mathbf{A}^H\mathbf{W}\mathbf{A})^{-1}\mathbf{A}^H\mathbf{W}\mathbf{b}.
$$

数值实现中通常不显式求逆，而是使用分解或迭代求解器。这个区别很重要：公式说明结构，算法决定稳定性。


一个简单直线拟合例子可以说明权重的作用。把模型

$$
y(t)=c+mt
$$

写成设计矩阵形式：

$$
\mathbf{A}=\begin{pmatrix}
1&t_1\\
1&t_2\\
\vdots&\vdots\\
1&t_N
\end{pmatrix},
\qquad
\mathbf{x}=\begin{pmatrix}c\\m\end{pmatrix}.
$$

若不同数据点的不确定度不同，权重矩阵的对角元应取 $w_i=1/\sigma_i^2$。误差条大的数据点对解的牵引较弱，误差条小的数据点对解的牵引较强。

![加权线性回归](figures/linear_algebra_weighted_fit.png)

**图 2.10.2** 加权与不加权线性回归的差异。权重改变了“最佳”解的位置，因为不同观测点的可靠性不同。干涉成像中的自然加权、均匀加权和鲁棒加权也体现了同一数学骨架。


### 2.10.5 卷积矩阵、循环矩阵与傅里叶基<a id='math:lin_alg:conv_toeplitz'></a>

离散卷积也可以写成矩阵乘法。对线性卷积，矩阵通常具有 Toeplitz 结构：每条对角线上的元素相同。对循环卷积，矩阵进一步成为 circulant matrix，每一行都是上一行的循环移位。长度为 $N$ 的循环卷积可写成

$$
\mathbf{r}=\mathbf{C}\mathbf{z},
$$

其中 $\mathbf{C}$ 由卷积核的循环移位组成。循环矩阵最重要的性质，是它可以被 DFT 矩阵对角化：

<a id='math:eq:circulant_diagonalization'></a><!--\label{math:eq:circulant_diagonalization}-->
$$
\mathbf{C}=\mathbf{F}^{-1}\boldsymbol{\Lambda}\mathbf{F}.
$$

这里 $\boldsymbol{\Lambda}$ 的对角元就是卷积核的 DFT。这个式子是离散卷积定理的矩阵版本：在原始域中卷积，对应在傅里叶基底中逐元素乘法。


### 2.10.6 干涉测量矩阵：$\mathbf{A}$、$\mathbf{A}^H$ 与 $\mathbf{A}^H\mathbf{A}$<a id='math:sec:interferometric_measurement_matrix'></a>

把一维天空离散成像素向量 $\mathbf{x}$，把若干空间频率样本堆叠成可见度向量 $\mathbf{b}$，最简测量模型可写成

$$
\mathbf{b}=\mathbf{A}\mathbf{x}+\boldsymbol{\epsilon},
$$

其中

$$
A_{pj}=e^{-2\pi i u_p l_j}
$$

表示第 $p$ 个 `uv` 样本对第 $j$ 个图像像素的响应。伴随算子 $\mathbf{A}^H$ 把可见度反投影回图像网格，形成脏图像；正规矩阵 $\mathbf{A}^H\mathbf{A}$ 则描述真实天空如何被点扩散函数模糊。

![一维测量矩阵示意](figures/linear_algebra_measurement_matrix.png)

**图 2.10.3** 一维干涉测量的矩阵视角。$\mathbf{A}$ 的行对应采样到的空间频率；$\mathbf{A}^H\mathbf{b}$ 是脏图像；$\mathbf{A}^H\mathbf{A}$ 的列给出脏波束形状；奇异值反映反演问题的病态程度。


这个视角把许多后续概念放在同一张图里。`uv` 覆盖改变的是 $\mathbf{A}$ 的行；脏波束来自 $\mathbf{A}^H\mathbf{A}$；加权相当于在残差空间中插入 $\mathbf{W}$；正则化则是在正规方程中加入额外约束。理解这些矩阵角色之后，下一节最小二乘与参数估计就会自然得多。


***

* 下一节：[2.11 最小二乘与参数估计](2_11_least_squares.ipynb)
